To run this notebook, please install:
- httpx2
- icechunk
- obstore
- obspec_utils
- virtualizarr
- virtual-tiff

This notebook stores a virtual tile collection. Its 36,000-pixel TIFF edges do not
align with 512-pixel chunks, so the raster cannot be concatenated into one virtual
Zarr grid. `run_time.ipynb` assembles a geographic mosaic lazily when reading;
raster bytes stay in the remote TIFFs.


In [ ]:
!pip install httpx2 icechunk obstore obspec_utils virtualizarr virtual-tiff

In [ ]:
import httpx2
import icechunk
import obstore
import warnings
import xarray as xr
from anyio import create_task_group
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_mfdataset
from virtual_tiff import VirtualTIFF
from imagecodecs.numcodecs import Lzw
from numcodecs.registry import register_codec

# VirtualTIFF uses Numcodecs to construct its native LZW codec.
# Register explicitly instead of relying on optional package entry points.
register_codec(Lzw)

In [ ]:
base_url = "https://data.hydrosheds.org/file/hydrosheds-v2/ACC/1s"
file_urls = []
for lat_10 in range(-6, 8 + 1):
    for lon_10 in range(-17, -4 + 1):
        lat = lat_10 * 10
        lon = lon_10 * 10
        if lat >= 0:
            _lat = f"n{lat}"
        else:
            _lat = f"s{-lat}"
        file_urls.append(f"{base_url}/{_lat}w{-lon}_ACC_1s_v2r0.tif")

async def check_url(client, url, file_urls):
    async with client.stream("GET", url) as response:
        if response.status_code == 404:
            file_urls.remove(url)


async with (
    httpx2.AsyncClient() as client,
    create_task_group() as tg,
):
    for url in file_urls:
        tg.start_soon(check_url, client, url, file_urls)

In [ ]:
store = obstore.store.HTTPStore.from_url(base_url)
registry = ObjectStoreRegistry({base_url: store})


reference_grid = None

def add_tile_coords(ds):
    data = next(iter(ds.data_vars.values()))
    global reference_grid
    attrs = data.attrs
    raster_x, raster_y, _, map_x, map_y, _ = attrs["model_tiepoint"]
    dx, dy, _ = attrs["model_pixel_scale"]
    if (attrs.get("geographic_type") != 4326 or attrs.get("raster_type") != 1
            or dx <= 0 or dy <= 0 or "model_transformation" in attrs):
        raise ValueError("Expected unrotated WGS84 PixelIsArea tiles")
    grid = (dx, dy)
    if reference_grid is None:
        reference_grid = grid
    elif grid != reference_grid:
        raise ValueError("All tiles must have the same pixel spacing")
    # Store the upper-left pixel corner even if the tiepoint is elsewhere.
    map_x -= raster_x * dx
    map_y += raster_y * dy

    return ds.expand_dims(tile=[f"{map_y:g}_{map_x:g}"]).assign_coords(
        tile_x=("tile", [map_x]),
        tile_y=("tile", [map_y]),
    )


# ipygis registers the TIFF LZW decoder in Zarrita explicitly.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=(
            r"^Imagecodecs codecs are not in the Zarr version 3 specification and "
            r"may not be supported by other zarr implementations\.$"
        ),
        category=UserWarning,
        module=r"^virtual_tiff(?:\.|$)",
    )
    ds = open_virtual_mfdataset(
        urls=file_urls,
        registry=registry,
        parser=VirtualTIFF(ifd=0),
        preprocess=add_tile_coords,
        combine="nested",
        concat_dim="tile",
    )

ds

In [ ]:
icechunk_path = "hydrosheds.icechunk"

# Allow Icechunk to resolve the referenced HTTP TIFF chunks.
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
  icechunk.VirtualChunkContainer(
      url_prefix=f"{base_url}/",
      store=icechunk.http_store(),
  )
)

# Create a persistent local Icechunk repository.
storage = icechunk.local_filesystem_storage(icechunk_path)
repo = icechunk.Repository.create(
  storage=storage,
  config=config,
  authorize_virtual_chunk_access={f"{base_url}/": icechunk.credentials.HttpAccess},
)

# Serialize and commit the virtual dataset.
session = repo.writable_session("main")
ds.vz.to_icechunk(session.store)
snapshot_id = session.commit("Add virtual HydroSHEDS tiles")

snapshot_id

In [ ]:
storage = icechunk.local_filesystem_storage(icechunk_path)
# Authorize remote TIFF reads again when reopening the repository.
repo = icechunk.Repository.open(
    storage=storage,
    authorize_virtual_chunk_access={f"{base_url}/": icechunk.credentials.HttpAccess},
)
session = repo.readonly_session(branch="main")

restored = xr.open_zarr(
  session.store,
  zarr_format=3,
  consolidated=False,
)

restored

In [ ]:
restored["0"].isel(tile=10, y=100, x=200).compute()